In [39]:
import pandas as pd
import re
import numpy as np

In [40]:
df = pd.read_csv("questionnaire_factor_associations_combined.csv")

In [41]:
def remove_k_all(x):

    x = str(x)

    # remove _K1, _K2, ...
    x = re.sub(r"_K\d+$", "", x)

    # remove trailing _K (important!)
    x = re.sub(r"_K$", "", x)

    return x

df["score_clean"] = df["score_base"].apply(remove_k_all)
df["factor_clean"] = df["factor_base"].apply(remove_k_all)

In [42]:
def weighted(group):

    assoc = group["association"].values
    weights = group["total_n"].values

    return pd.Series({
        "association": np.average(assoc, weights=weights),
        "total_n": weights.sum(),
        "num_points": len(group),
        "factor_type": group["factor_type"].iloc[0]
    })

In [43]:
merged = (
    df
    .groupby([
        "questionnaire",
        "score_clean",
        "factor_clean"
    ])
    .apply(weighted)
    .reset_index()
)

C:\Users\veron\AppData\Local\Temp\ipykernel_38424\1196506708.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted)


In [44]:
merged = merged.rename(columns={
    "score_clean": "score",
    "factor_clean": "factor"
})

In [45]:
merged["abs_assoc"] = merged["association"].abs()

merged = merged.sort_values(
    ["questionnaire", "abs_assoc"],
    ascending=[True, False]
)

In [46]:
for q in merged["questionnaire"].unique():

    print("\n" + "=" * 80)
    print(q)
    print("=" * 80)

    print(
        merged[merged["questionnaire"] == q]
        .head(20)[
            ["score", "factor", "association", "total_n"]
        ]
    )



chiq_result
          score                          factor  association  total_n
8   chiq_result             days_lost_household     0.552998      205
14  chiq_result  headache_days_severe_per_month     0.484891      205
13  chiq_result         headache_days_per_month     0.439944      205
10  chiq_result                 days_medication     0.418405      205
6   chiq_result                     days_doctor     0.385489      205
9   chiq_result                  days_lost_work     0.382598      205
23  chiq_result           working_hours_reduced     0.360847      205
17  chiq_result                       intensity     0.338243      205
19  chiq_result                promotion_waived     0.269827      205
5   chiq_result                         days_ER     0.187254      205
3   chiq_result                        children    -0.152343      118
7   chiq_result                   days_hospital     0.113610      205
0   chiq_result                       birthyear     0.097736      205
15  chi

In [47]:
merged.to_csv(
    "questionnaire_factor_associations_FINAL_MERGED.csv",
    index=False
)

print("\nSaved FINAL merged dataset.")


Saved FINAL merged dataset.
